# Homework 1

## Problem 1

Make a tuple containing natural numbers, the square of which is a multiple of 3, 4, but not a multiple of 8 and not exceeding 12345.

In [17]:
tuple(n for n in range(6, int(12345**0.5) + 1, 12))

(6, 18, 30, 42, 54, 66, 78, 90, 102)

## Problem 2


Write a function that takes a two-dimensional array and a string as input and returns an array rotated 90 degrees counterclockwise if the string 'left' was passed, and clockwise if the string 'right' was passed.

Example for input: $\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{bmatrix}$.\
If the string 'left' is passed, the function should return $\begin{bmatrix} 3 & 6 & 9 \\ 2 & 5 & 8 \\ 1 & 4 & 7 \end{bmatrix}$, and if the string 'right' is passed, the function should return $\begin{bmatrix} 7 & 4 & 1 \\ 8 & 5 & 2 \\ 9 & 6 & 3 \end{bmatrix}$.

Do not use numpy and nested loops. Use list comprehensions and built-in function `zip`.

In [18]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

def rotate(matrix, direction):
    if direction == "right":
        return [list(row) for row in zip(*matrix[::-1])]
    elif direction == "left":
        return [list(row) for row in zip(*matrix)][::-1]

rotate(matrix, 'left')
#rotate(matrix, 'right')

[[3, 6, 9], [2, 5, 8], [1, 4, 7]]

## Problem 3

Write a function that takes a string as input and returns a dictionary containing the number of occurrences of each character in the string. Use method `count` of strings. Implement this function in one line using $\lambda$-function.

Example for the string 'hello, world!': {'h': 1, 'e': 1, 'l': 3, 'o': 2, ',': 1, ' ': 1, 'w': 1, 'r': 1, 'd': 1, '!': 1}.

In [19]:
char_count = lambda s: {c: s.count(c) for c in set(s)}
print(char_count('hello, world!'))

{'o': 2, 'e': 1, '!': 1, ',': 1, 'r': 1, 'h': 1, 'w': 1, 'l': 3, ' ': 1, 'd': 1}


## Problem 4

### Implementing a Library Management System

#### Description

You are required to design and implement a system for managing books and users in a library. The system should allow for the management of books (adding, deleting, searching by various criteria) and users (registration, deletion, searching), as well as tracking the history of interactions between them (issuing and returning books).

#### Tasks

1. **`Book` Class**:
   - Attributes: title, author, year of publication, ISBN, number of copies.
   - Methods: constructor, methods to get information about the book, method to change the number of copies (when issuing and returning books).

2. **`User` Class**:
   - Attributes: user name, library card number, list of borrowed books.
   - Methods: constructor, methods for user registration, methods for adding and removing books from the borrowed list.

3. **`Library` Class**:
   - Attributes: list of books, list of users, transaction history (who, when, which book was borrowed and returned).
   - Methods: constructor, methods for adding and deleting books and users, methods for issuing and returning books, searching for books and users by various criteria, method to display the transaction history.

#### Assignment

1. Implement the `Book`, `User`, and `Library` classes with the specified attributes and methods.
2. Create several books and users, and add them to the library system.
3. Implement scenarios for issuing books to users and their return.
4. Display the transaction history to show how books were issued and returned.


In [20]:
from datetime import datetime

# ------------------- Book Class -------------------
class Book:
    def __init__(self, title, author, year, isbn, copies):
        self.title = title
        self.author = author
        self.year = year
        self.isbn = isbn
        self.copies = copies  # сколько копий в библиотеке

    def info(self):
        return f"{self.title} by {self.author}, {self.year}, ISBN: {self.isbn}, Copies: {self.copies}"

    def change_copies(self, delta):
        """Изменяет количество копий при выдаче или возврате"""
        self.copies += delta

# ------------------- User Class -------------------
class User:
    def __init__(self, name, card_number):
        self.name = name
        self.card_number = card_number
        self.borrowed_books = []  # список объектов Book

    def borrow_book(self, book):
        self.borrowed_books.append(book)

    def return_book(self, book):
        if book in self.borrowed_books:
            self.borrowed_books.remove(book)

    def info(self):
        borrowed_titles = [b.title for b in self.borrowed_books]
        return f"{self.name} (Card {self.card_number}) | Borrowed: {borrowed_titles}"

# ------------------- Library Class -------------------
class Library:
    def __init__(self):
        self.books = []  # список объектов Book
        self.users = []  # список объектов User
        self.history = []  # список транзакций: {"user":..., "book":..., "action":..., "time":...}

    # ---------------- Books ----------------
    def add_book(self, book):
        self.books.append(book)

    def delete_book(self, isbn):
        self.books = [b for b in self.books if b.isbn != isbn]

    def search_books(self, **criteria):
        """Можно искать по title, author, year, isbn"""
        results = self.books
        for key, value in criteria.items():
            results = [b for b in results if getattr(b, key) == value]
        return results

    # ---------------- Users ----------------
    def add_user(self, user):
        self.users.append(user)

    def delete_user(self, card_number):
        self.users = [u for u in self.users if u.card_number != card_number]

    def search_users(self, **criteria):
        """Можно искать по name, card_number"""
        results = self.users
        for key, value in criteria.items():
            results = [u for u in results if getattr(u, key) == value]
        return results

    # ---------------- Transactions ----------------
    def issue_book(self, card_number, isbn):
        user = self.search_users(card_number=card_number)
        book = self.search_books(isbn=isbn)
        if not user or not book:
            print("Пользователь или книга не найдены!")
            return
        user = user[0]
        book = book[0]
        if book.copies <= 0:
            print(f"Все копии книги '{book.title}' выданы!")
            return
        book.change_copies(-1)
        user.borrow_book(book)
        self.history.append({
            "user": user.name,
            "book": book.title,
            "action": "issued",
            "time": datetime.now()
        })
        print(f"Книга '{book.title}' выдана пользователю {user.name}.")

    def return_book(self, card_number, isbn):
        user = self.search_users(card_number=card_number)
        book = self.search_books(isbn=isbn)
        if not user or not book:
            print("Пользователь или книга не найдены!")
            return
        user = user[0]
        book = book[0]
        if book not in user.borrowed_books:
            print(f"Пользователь {user.name} не брал эту книгу.")
            return
        book.change_copies(1)
        user.return_book(book)
        self.history.append({
            "user": user.name,
            "book": book.title,
            "action": "returned",
            "time": datetime.now()
        })
        print(f"Книга '{book.title}' возвращена пользователем {user.name}.")

    # ---------------- History ----------------
    def show_history(self):
        for record in self.history:
            t = record["time"].strftime("%Y-%m-%d %H:%M:%S")
            print(f"{t} | {record['user']} {record['action']} '{record['book']}'")

# ------------------- Пример использования -------------------
if __name__ == "__main__":
    # создаём библиотеку
    lib = Library()

    # добавляем книги
    b1 = Book("1984", "George Orwell", 1949, "12345", 3)
    b2 = Book("Python 101", "Michael Driscoll", 2019, "23456", 2)
    b3 = Book("Harry Potter", "J.K. Rowling", 2007, "34567", 1)
    lib.add_book(b1)
    lib.add_book(b2)
    lib.add_book(b3)

    # добавляем пользователей
    u1 = User("Alice", 1)
    u2 = User("Bob", 2)
    lib.add_user(u1)
    lib.add_user(u2)

    # выдаём книги
    lib.issue_book(1, "12345")  # Alice берёт "1984"
    lib.issue_book(2, "23456")  # Bob берёт "Python 101"
    lib.issue_book(1, "34567")  # Alice берёт "Harry Potter"

    # возвращаем книги
    lib.return_book(1, "12345")  # Alice возвращает "1984"

    # смотрим историю
    print("\nИстория транзакций:")
    lib.show_history()

Книга '1984' выдана пользователю Alice.
Книга 'Python 101' выдана пользователю Bob.
Книга 'Harry Potter' выдана пользователю Alice.
Книга '1984' возвращена пользователем Alice.

История транзакций:
2026-02-24 00:44:32 | Alice issued '1984'
2026-02-24 00:44:32 | Bob issued 'Python 101'
2026-02-24 00:44:32 | Alice issued 'Harry Potter'
2026-02-24 00:44:32 | Alice returned '1984'


## Problem 5*

Explain why list `b` changes after the execution of the following code:

```python
a = [1, 2, 3]
b = a 
a[0] = 4
print(b)
```

> Write your answer in markdown cell after:

Когда выполняется:

`
a = [1, 2, 3]
b = a
`

переменные a и b ссылаются на один и тот же список в памяти. То есть b это просто другое имя того же объекта, а не отдельная копия.

Далее:
`
a[0] = 4
`
мы изменяем сам список. Так как b ссылается на этот же объект, изменения видны и через b.

Поэтому вывод:
`
print(b)
`
будет:
`
[4, 2, 3]
`
Важно: присвоение одного списка другому не создаёт копию, а копирует ссылку на один объект.

Чтобы b был независимым списком, нужно создать копию:
`
b = a.copy()
`

## Problem 6*

Let 
$$A = \sum_{i=1}^{10000} \frac{1}{i^2},\quad B=\sum_{i=10000}^{1} \frac{1}{i^2}.$$
Calculate the values of $A$ and $B$ and compare them. What do you observe? Explain why this happens. What is the best way to calculate the value of $\sum\limits_{i=1}^{10000} \dfrac{1}{i^2}$?

In [1]:
A = sum(1/i**2 for i in range(1, 10001))        # суммируем от 1 до 10000
B = sum(1/i**2 for i in range(10000, 0, -1))    # суммируем от 10000 до 1
print(A)
print(B)

1.6448340718480652
1.6448340718480596


Все дело в ошибках округления. Если начинать сумму с маленьких чисел то точность выше, а если с больших то точность пададет со временем. Лучше складывать с маленьких, так точнее.